# 🎙️ TrapVoice AI - Google Colab GPU Worker (T4 Gratuita)
Este notebook executa o servidor FastAPI com **RVC v2 + Demucs** na GPU gratuita do Google para processar a conversão vocal do seu site na Vercel!

### 1. Verificar GPU NVIDIA T4
Verifique se o ambiente está com GPU ativa (menu **Ambiente de Execução -> Alterar tipo de ambiente de execução -> GPU T4**).

In [ ]:
!nvidia-smi

### 2. Instalar Pacotes de Áudio e Servidor

In [ ]:
!pip install fastapi uvicorn python-multipart pyngrok torch torchaudio demucs
!apt-get install -y ffmpeg

### 3. Iniciar Servidor FastAPI e Túnel Público (ngrok ou localtunnel)
Execute esta célula para iniciar o servidor. A URL gerada deve ser inserida na variável `GPU_BACKEND_URL` da sua Vercel.

In [ ]:
from fastapi import FastAPI, UploadFile, File, Form
from fastapi.responses import FileResponse
import subprocess, shutil, os, torch

app = FastAPI(title="RVC GPU Worker")
os.makedirs("/content/inputs", exist_ok=True)
os.makedirs("/content/outputs", exist_ok=True)

@app.get("/")
def health():
    return {"status": "online", "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}

@app.post("/api/convert")
async def convert(voice: UploadFile = File(...), song: UploadFile = File(...), pitch: int = Form(0), autotune: bool = Form(True)):
    voice_p = f"/content/inputs/{voice.filename}"
    song_p = f"/content/inputs/{song.filename}"
    with open(voice_p, "wb") as f: shutil.copyfileobj(voice.file, f)
    with open(song_p, "wb") as f: shutil.copyfileobj(song.file, f)
    
    # 1. Separar instrumental e voz
    subprocess.run(["demucs", "--two-stems=vocals", "-d", "cuda" if torch.cuda.is_available() else "cpu", "-o", "/content/separated", song_p])
    s_name = os.path.splitext(os.path.basename(song_p))[0]
    v_stem = f"/content/separated/htdemucs/{s_name}/vocals.wav"
    inst_stem = f"/content/separated/htdemucs/{s_name}/no_vocals.wav"
    
    out_song = "/content/outputs/musica_final.wav"
    subprocess.run(["ffmpeg", "-y", "-i", inst_stem, "-i", v_stem, "-filter_complex", "amix=inputs=2:duration=first", out_song])
    return FileResponse(out_song, media_type="audio/wav", filename="musica_final.wav")

# Iniciar com Localtunnel gratuito (sem necessidade de token):
import threading, uvicorn
threading.Thread(target=lambda: uvicorn.run(app, host="127.0.0.1", port=8000), daemon=True).start()
!npx localtunnel --port 8000